figure out how to deal with ztf data

In [66]:
from antares_devkit.models import BaseFilter
from antares_devkit.models import DevKitLocus
from antares_client import search
import numpy as np
import matplotlib.pyplot as plt
from astropy.time import Time
import pandas as pd
import math
from astropy.coordinates import Angle, SkyCoord
import astropy.units as u

t_now = Time.now().mjd

In [5]:
ztf_tester = search.get_by_ztf_object_id("ZTF18abtimvz")
oid_dev = ztf_tester.to_devkit()
ztf_locus = DevKitLocus.model_validate(oid_dev)

In [62]:
ztf_locus.timeseries

<bound method DevKitLocus.timeseries of DevKitLocus(id='ANT2020bppyw', ra=299.78528306999476, dec=-8.384502617330384, alerts=[DevkitAlert(id='ztf_upper_limit:ZTF18abtimvz-590337404415', mjd=58344.3374073999, properties={'ztf_jd': 2458344.8374074, 'ztf_fid': 1, 'ztf_pid': 590337404415, 'ztf_diffmaglim': 20.434900283813477, 'ztf_pdiffimfilename': '/ztf/archive/sci/2018/0814/337407/ztf_20180814337407_000387_zg_c12_o_q1_scimrefdiffimg.fits.fz', 'ztf_programpi': 'Kulkarni', 'ztf_programid': 1, 'ztf_rbversion': 't12_f5_c3', 'ant_mjd': 58344.3374073999, 'ant_time_received': 1594247233, 'ant_input_msg_time': 1594243324, 'ant_passband': 'g', 'ant_maglim': 20.434900283813477, 'ant_survey': 2}, non_persistent_properties={}, grav_wave_events=[]), DevkitAlert(id='ztf_upper_limit:ZTF18abtimvz-593328954415', mjd=58347.32895829994, properties={'ztf_jd': 2458347.8289583, 'ztf_fid': 1, 'ztf_pid': 593328954415, 'ztf_diffmaglim': 20.329999923706055, 'ztf_pdiffimfilename': '/ztf/archive/sci/2018/0817/32895

In [77]:
ztf_locus.properties

{'ztf_object_id': 'ZTF18abtimvz',
 'num_alerts': 864,
 'num_mag_values': 667,
 'newest_alert_id': 'ztf_candidate:3446413354415015011',
 'brightest_alert_id': 'ztf_candidate:3426456634415015011',
 'ztf_ssnamenr': 'null',
 'oldest_alert_id': 'ztf_candidate:612229654415015014',
 'oldest_alert_magnitude': 19.30623435974121,
 'oldest_alert_observation_time': 58366.22965280013,
 'newest_alert_magnitude': 16.67751121520996,
 'newest_alert_observation_time': 61200.41335650021,
 'brightest_alert_magnitude': 16.489442825317383,
 'brightest_alert_observation_time': 61180.45663189981,
 'is_corrected': 'true',
 'survey': {'ztf': {'id': ['ZTF18abtimvz', 'ZTF22aajtwau'],
   'rcid': [5, 44],
   'field': [387, 1432],
   'ssnamenr': ['2006SU180', '533438', 'null']}},
 'feature_amplitude_magn_r': 1.0206060409545898,
 'feature_anderson_darling_normal_magn_r': 1.0089801232604636,
 'feature_beyond_1_std_magn_r': 0.2857142857142857,
 'feature_beyond_2_std_magn_r': 0.050793650793650794,
 'feature_cusum_magn_r

In [9]:
ztf_alerts = []
for alert in ztf_locus.alerts:
    if alert.properties['ant_survey'] != 1:
        continue
    ztf_alerts.append(alert)

In [27]:
ztf_alerts[0].properties 

{'ztf_jd': 2458366.7296528,
 'ztf_fid': 1,
 'ztf_pid': 612229654415,
 'ztf_diffmaglim': 20.817697525024414,
 'ztf_pdiffimfilename': 'ztf_20180905229641_000387_zg_c12_o_q1_scimrefdiffimg.fits',
 'ztf_programpi': 'Kulkarni',
 'ztf_programid': 1,
 'ztf_candid': 612229654415015014,
 'ztf_isdiffpos': 't',
 'ztf_tblid': 14,
 'ztf_nid': 612,
 'ztf_rcid': 44,
 'ztf_field': 387,
 'ztf_xpos': 98.91169738769531,
 'ztf_ypos': 1249.1507568359375,
 'ztf_ra': 299.7852777,
 'ztf_dec': -8.3843448,
 'ztf_magpsf': 19.30623435974121,
 'ztf_sigmapsf': 0.11770743876695633,
 'ztf_chipsf': 2.472749710083008,
 'ztf_magap': 19.598100662231445,
 'ztf_sigmagap': 0.15320000052452087,
 'ztf_distnr': 0.2496344894170761,
 'ztf_magnr': 16.505001068115234,
 'ztf_sigmagnr': 0.017000000923871994,
 'ztf_chinr': 0.6909999847412109,
 'ztf_sharpnr': -0.02500000037252903,
 'ztf_sky': 0.015572070144116879,
 'ztf_magdiff': 0.2918660044670105,
 'ztf_fwhm': 1.3297970294952393,
 'ztf_classtar': 0.9269999861717224,
 'ztf_mindtoedge

magnr is acting as the template mag (i think), 
ztf_fid: 1 - 'g', 2 - 'R', 3 - 'i'

below, i am using the instructions in [the ztf user guide](https://irsa.ipac.caltech.edu/data/ZTF/docs/ztf_zfps_userguide.pdf) (p. 12) to compute the fluxes and then mags of the oid and associated pannstars ref oid --- actually i dont think this can be used, is uses variables that we dont have access to (like zpdiff - the zero point used to det the mag values for the forced phot and pannstars nr obj)

In [186]:
ztf_mags = {'g': 0, 'R': 0, 'i': 0}
band = ['x', 'g', 'R', 'i']
magzp = [0, 26.325, 26.275, 25.660]
for alert in ztf_alerts:
    if alert.properties['ant_survey'] != 1:
        continue

    band_id = alert.properties['ztf_fid']
    obs_date = float(alert.properties['ant_mjd'])
    
    pnr_flux = 10**(0.4*(magzp[band_id] - alert.properties['ztf_magnr']))
    diff_flux = 10**(0.4*(magzp[band_id] - alert.properties['ztf_magpsf']))
    
    if alert.properties['ztf_isdiffpos'] == 'f' or alert.properties['ztf_isdiffpos'] == 0:
        oid_flux = pnr_flux - diff_flux
        pn = -1
    else:
        oid_flux = pnr_flux + diff_flux
        pn = 1
    
    pnr_mag = float(alert.properties['ztf_magnr'])
    oid_mag = float(-2.5*np.log10(oid_flux) + magzp[band_id])
    diff_mag = abs(oid_mag - pnr_mag)

    pnr_sigma = pnr_mag * pnr_flux / 1.0857
    oid_sigma = oid_mag * oid_flux / 1.0857
    
    oid_snr = oid_sigma / oid_flux
    ant_mag = alert.properties['ant_mag']
    data = [obs_date, ant_mag, oid_mag, pnr_mag, diff_mag, pn, oid_snr]
    if ztf_mags[band[band_id]] == 0:
        df = pd.DataFrame([data], columns = ['mjd', 'ant_mag', 'oid_mag', 'template_mag', 'diff_mag', 'pos/neg', 'snr'])
        ztf_mags[band[band_id]] = {'data': df, 'variance': 0, 'hyper_v': 0}
    else:
        data_df = pd.DataFrame([data], columns = ['mjd', 'ant_mag', 'oid_mag', 'template_mag', 'diff_mag', 'pos/neg', 'snr'])
        
        current_data = ztf_mags[band[band_id]]['data']
        ztf_mags[band[band_id]]['data'] = pd.concat([current_data, data_df]).drop_duplicates().reset_index(drop=True)

# for band in ztf_mags:
#     if len(ztf_mags[band]) < 2:
#         continue
#     print(np.var(ztf_mags[band]['mag']))

In [187]:
ztf_mags

{'g': {'data':               mjd    ant_mag    oid_mag  template_mag  diff_mag  pos/neg  \
  0    58366.229653  19.306234  16.425701     16.505001  0.079300        1   
  1    58372.256505  18.841627  16.385609     16.505001  0.119392        1   
  2    58428.149699  18.855658  16.387061     16.505001  0.117940        1   
  3    58593.482928  18.352301  16.323067     16.504999  0.181932        1   
  4    58607.419005  19.035299  16.404237     16.504999  0.100762        1   
  ..            ...        ...        ...           ...       ...      ...   
  315  61182.446134  18.631445  16.361724     16.505001  0.143277        1   
  316  61184.423276  17.511118  16.142888     16.505001  0.362113        1   
  317  61186.436169  17.597569  16.166714     16.505001  0.338287        1   
  318  61193.397141  18.211990  16.300203     16.505001  0.204798        1   
  319  61199.421690  17.744411  16.204120     16.505001  0.300881        1   
  
             snr  
  0    15.129134  
  1    15.

In [37]:
diff_flux = 10**(0.4*(26.325-19.30623435974121))
pns_flux = 10**(0.4*(26.325-16.505001068115234))
sci_flux = diff_flux + pns_flux
sci_mag = -2.5*np.log10(sci_flux) + 26.325
print(sci_mag)

16.42570084124425


okay!! this seems to work well for the ztf data from the oid that i passed to it. now i want to figure out how to combine this with the lsst alert stream so an oid w both could be detected using either survey. the main issue with this is that both surveys have g, r/R, and i bands but they have slightly different transmissions and so need to stay seperated. i need to figure out the best way to store all of this data and keep it organized. 

In [266]:
def lsst_alert(alert):
    obs_date = float(alert.properties['lsst_diaSource_midpointMjdTai'])
    oid_mag = float(-2.5*np.log10(alert.properties['lsst_diaSource_scienceFlux']) + 31.4)
    temp_mag = float(-2.5*np.log10(alert.properties['lsst_diaSource_templateFlux']) + 31.4)
    diff_mag = float(temp_mag - oid_mag)
    snr = float(alert.properties['lsst_diaSource_snr'])
    
    bandpass = alert.properties['lsst_diaSource_band']
    band = 'lsst_' + str(bandpass)

    if alert.properties['lsst_diaSource_isNegative'] == True:
        pos_or_neg = -1
    else:
        pos_or_neg = 1
        
    return {band: [obs_date, oid_mag, temp_mag, diff_mag, pos_or_neg, snr]}


def ztf_alert1(alert):
    bands = ['x', 'g', 'R', 'i']
    magzp = [0, 26.325, 26.275, 25.660]
    
    band_id = alert.properties['ztf_fid']
    obs_date = float(alert.properties['ant_mjd'])
    ant_mag = alert.properties['ant_mag']
    pnr_flux = 10**(0.4*(magzp[band_id] - alert.properties['ztf_magnr']))
    diff_flux = 10**(0.4*(magzp[band_id] - alert.properties['ztf_magpsf']))
    
    if alert.properties['ztf_isdiffpos'] == 'f' or alert.properties['ztf_isdiffpos'] == 0:
        oid_flux = pnr_flux - diff_flux
        pos_or_neg = -1
    else:
        oid_flux = pnr_flux + diff_flux
        pos_or_neg = 1
    
    pnr_mag = float(alert.properties['ztf_magnr'])
    oid_mag = float(-2.5*np.log10(oid_flux) + magzp[band_id])
    diff_mag = abs(oid_mag - pnr_mag)

    oid_snr = float(1.0857 / alert.properties['ztf_sigmapsf'])

    band = 'ztf_' + str(bands[band_id])
    return {band: [obs_date, oid_mag, pnr_mag, diff_mag, pos_or_neg, oid_snr]}

In [280]:
def oid_var_check(alerts, day_lim, var_thresh):
    oid = {'lsst_g': 0, 'lsst_r': 0, 'lsst_i': 0, 'lsst_z': 0, 'lsst_u': 0, 'lsst_y': 0, 
               'ztf_g': 0, 'ztf_R': 0, 'ztf_i': 0}
    latest_mjd = 0
    
    for alert in alerts:
        if alert.properties['ant_survey'] == 4:
            comp = lsst_alert(alert)
        elif alert.properties['ant_survey'] == 1:
            ztf_pass = ztf_quality(alert)
            if ztf_pass == True:
                comp = ztf_alert2(alert)
            else:
                continue
        else:
            continue
            
        band = [*comp][0]
        
        if latest_mjd < comp[band][0]:
            latest_mjd = comp[band][0]
        
        if oid[band] == 0:
            oid_df = pd.DataFrame([comp[band]], columns = ['mjd', 'oid_mag', 'template_mag', 'diff_mag', 'pos/neg', 'snr'])
            oid[band] = {'data': oid_df, 'variance': 0, 'var_diff_rat': 0}
            continue
        else:
            data_df = pd.DataFrame([comp[band]], columns = ['mjd', 'oid_mag', 'template_mag', 'diff_mag', 'pos/neg', 'snr'])
        
            current_data = oid[band]['data']
            oid[band]['data'] = pd.concat([current_data, data_df]).drop_duplicates().reset_index(drop=True)
        
    var_check = 0
    for band in oid:
        if oid[band] == 0:
            continue

        recent_obs = oid[band]['data'][oid[band]['data']['mjd'] >= float(latest_mjd) - day_lim].index.tolist()

        if len(recent_obs) > 2: #if +3 alerts w/in day_lim of latest_mjd, then we look at the variance + hyper_v (only including alerts w/in time frame)
            temp_var = float(np.var(oid[band]['data']['template_mag']))
            if temp_var >= 0.01:
                continue
            
            here_var = float(np.var(oid[band]['data']['oid_mag'][recent_obs[0]:]))
            here_diffs = float(np.median(oid[band]['data']['diff_mag'][recent_obs[0]:]))
            var_diff = float(here_diffs/here_var)
                
            oid[band]['variance'] = here_var
            oid[band]['var_diff_rat'] = var_diff
        
            if here_var >= var_thresh:
                var_check += 1

    if var_check > 0:
        return oid


i want to test to see if this works for oids w lsst and ztf alerts!

In [171]:
mag_max = 22
mag_min = 18

q_lsst_magcut = {
    "query": {
        "bool": {
            "filter": [
                { "range": { "properties.newest_alert_observation_time": { "gte": t_now-50} } },
                { "range": { "properties.newest_alert_magnitude": { "lte": mag_max, "gte": mag_min} } },
                { "exists": { "field": "properties.survey.lsst" } },
             ]
        }
    }
}

In [224]:
oids_var = {}
passed = 0
for locus in search.search(q_lsst_magcut):
    oid_dv = locus.to_devkit()
    oid_locus = DevKitLocus.model_validate(oid_dv)

    high_var_oid = oid_var_check(oid_locus.alerts, 20, 0.2)

    if high_var_oid == None:
        passed += 1
        continue

    oids_var[oid_dv['id']] = high_var_oid
    
    if len(oids_var) >= 5:
        break
        
print(f'{passed} oids were passed\n{oids_var}')

/var/folders/ym/gbj1z46s0wg5zdmkkq0xpptr0000gs/T/ipykernel_70586/1174509041.py:37: RuntimeWarning: invalid value encountered in log10
  oid_mag = float(-2.5*np.log10(oid_flux) + magzp[band_id])


101 oids were passed
{'ANT2020b3cmq': {'lsst_g': 0, 'lsst_r': {'data':             mjd    oid_mag  template_mag  diff_mag  pos/neg         snr
0  61183.393621  17.367339     20.234066  2.866727        1  339.524475, 'variance': 0, 'hyper_v': 0}, 'lsst_i': 0, 'lsst_z': 0, 'lsst_u': 0, 'lsst_y': 0, 'ztf_g': {'data':               mjd    oid_mag  template_mag  diff_mag  pos/neg        snr
0    58637.480764  21.225650         18.57  2.655650       -1   7.226774
1    58649.356065  20.073490         18.57  1.503491       -1   5.667380
2    58654.313912  17.464972         18.57  1.105028        1  11.618848
3    58661.441146  21.724614         18.57  3.154614       -1  11.759035
4    58667.335474  19.310681         18.57  0.740681       -1   7.660070
..            ...        ...           ...       ...      ...        ...
296  61196.402512  22.897279         18.57  4.327279       -1  11.502337
297  61197.438866  20.536688         18.57  1.966688       -1  10.359536
298  61199.333160  20.53173

there is definitely smth wrong w how im calcing the oid mag, the first oid that passes [ANT2020b3cmq](https://antares.noirlab.edu/loci/ANT2020b3cmq) shows mags from like 17.5 up to nearly 23, even tho in the lightcurve on antares' pg doesnt even show it going lower than 20 a single time. wtf????

idk if putting them together like this will work, this is so much more sensitive to the ztf data, which varies a lot? idk it seems that this threshold doesn't pull out as many oids w interesting lightcurves when using ztf data

In [193]:
ztf_tester = search.get_by_id("ANT2020b3cmq")
oid_dev = ztf_tester.to_devkit()
ztf_locus = DevKitLocus.model_validate(oid_dev)

In [248]:
ztf_locus.alerts[8].properties['ztf_isdiffpos'] == 'f'

True

In [250]:
bands = ['x', 'g', 'R', 'i']
magzp = [0, 26.325, 26.275, 25.660]

for alert in ztf_locus.alerts:
    if alert.properties['ant_survey'] != 1:
        continue
        
    band_id = alert.properties['ztf_fid']
    obs_date = float(alert.properties['ant_mjd'])
    ant_mag = alert.properties['ant_mag']
    pnr_flux = 10**(0.4*(magzp[band_id] - alert.properties['ztf_magnr']))
    diff_flux = 10**(0.4*(magzp[band_id] - alert.properties['ztf_magpsf']))
    
    if alert.properties['ztf_isdiffpos'] == 'f' or alert.properties['ztf_isdiffpos'] == 0:
        oid_flux = pnr_flux - diff_flux
        pos_or_neg = alert.properties['ztf_isdiffpos']
    else:
        oid_flux = pnr_flux + diff_flux
        pos_or_neg = alert.properties['ztf_isdiffpos']
    
    pnr_mag = float(alert.properties['ztf_magnr'])
    oid_mag = float(-2.5*np.log10(oid_flux) + magzp[band_id])
    diff_mag = abs(oid_mag - pnr_mag)

    oid_snr = float(1.0857 / alert.properties['ztf_sigmapsf'])

    if bands[band_id] == 'g':
        if alert.properties['ztf_isdiffpos'] == 'f' or alert.properties['ztf_isdiffpos'] == 0:
            print(f"{obs_date, oid_mag, pnr_mag, diff_mag, alert.properties['ztf_magpsf']}\n{pos_or_neg}: diff ({diff_flux}) = pns ref ({pnr_flux}) - sci oid ({oid_flux})\n")
        else:
            print(f"{obs_date, oid_mag, pnr_mag, diff_mag, alert.properties['ztf_magpsf']}\n{pos_or_neg}: diff ({diff_flux}) = sci oid ({oid_flux}) -  pns ref({pnr_flux})\n")

(58637.480763900094, 21.225649969677903, 18.56999969482422, 2.6556502748536843, 18.668399810791016)
f: diff (1155.1545037328162) = pns ref (1264.7367029598558) - sci oid (109.58219922703961)

(58649.35606480017, 20.07349026030874, 18.56999969482422, 1.5034905654845225, 18.88290023803711)
f: diff (948.0689095273461) = pns ref (1264.7367029598558) - sci oid (316.6677934325097)

(58654.31391200004, 17.464971791074376, 18.56999969482422, 1.1050279037498427, 17.951900482177734)
t: diff (2234.8058890907673) = sci oid (3499.542592050623) -  pns ref(1264.7367029598558)

(58661.441145800054, 21.724614100266503, 18.56999969482422, 3.1546144054422847, 18.631099700927734)
f: diff (1195.5290119714696) = pns ref (1264.7367029598558) - sci oid (69.20769098838628)

(58667.33547449997, 19.31068066054348, 18.56999969482422, 0.7406809657192603, 19.3346004486084)
f: diff (625.4027989332343) = pns ref (1264.7367029598558) - sci oid (639.3339040266216)

(58673.31432869984, 17.98482510616018, 18.569999694824

/var/folders/ym/gbj1z46s0wg5zdmkkq0xpptr0000gs/T/ipykernel_70586/3172057793.py:22: RuntimeWarning: invalid value encountered in log10
  oid_mag = float(-2.5*np.log10(oid_flux) + magzp[band_id])


looking at the various mag values, a pattern emerges showing that this oid is using 2 mags for the pnr_mag, which should not be happening. the difference mag are based on the pannstars reference object, and if they are using multiple pns ref oids, then the difference mag is basically useless. so, i think i need to make another parameter that checks the variance of the pns mags and makes sure that they dont vary by much. honestly, this is a good check for all bands, not just ztf bands, since if the template varies a lot for anything then we can't really use the difference info at all. im going to go up and add a check for this :))

actually that wasnt the issue, the different ones are from different bands so ofc theyd be different. so something is truely wrong with how im determining the oid mag. when i print just one band (the g band in this case), it becomes even more obvious since there are some mags that show as nan and oid fluxes that are negative. this sucks.

i printed the information out in a way that makes it easy for me to compare each observation with eachother and compare tot eh data in the antares gui. in doing so, i noticed that antares used magpsf as the mag of the oid itself. i thought this was the difference mag, but it is super unclear in the properties description. if this is true, then the alert packets actually dont have a variable for diff mag??? im going to make another version of this below, where i use magpsf as the oid_mag, and still use magnr (pns ref mag) as the template_mag and see what happens.

In [264]:
bands = ['x', 'g', 'R', 'i']
magzp = [0, 26.325, 26.275, 25.660]

for alert in ztf_locus.alerts:
    if alert.properties['ant_survey'] != 1:
        continue
    elif alert.properties['ztf_rb'] < 0.65:
        continue
    elif alert.properties['ztf_nbad'] != 0:
        continue
    elif alert.properties['ztf_fwhm'] > 5:
        continue
    elif alert.properties['ztf_elong'] > 1.2:
        continue
    elif abs(alert.properties['ztf_magdiff']) > 0.1:
        continue

    band_id = alert.properties['ztf_fid']
    obs_date = float(alert.properties['ant_mjd'])
    ant_mag = float(alert.properties['ant_mag'])
    oid_mag = float(alert.properties['ztf_magpsf'])
    temp_mag = float(alert.properties['ztf_magnr'])

    temp_flux = 10**(0.4*(magzp[band_id] - temp_mag))
    oid_flux = 10**(0.4*(magzp[band_id] - oid_mag))
    
    oid_snr = float(1.0857 / alert.properties['ztf_sigmapsf'])
    pos_or_neg = alert.properties['ztf_isdiffpos']
    
    if pos_or_neg == 'f' or pos_or_neg == 0:
        diff_flux = temp_flux - oid_flux
        diff_mag = temp_mag - oid_mag
    else:
        diff_flux = oid_flux - temp_flux
        diff_mag = oid_mag - temp_mag
    
    if bands[band_id] == 'g':
        # print(f'{obs_date, oid_mag, temp_mag, diff_mag, pos_or_neg}\n')
        print(f'{obs_date, ant_mag, oid_mag, temp_mag, diff_mag}\n{pos_or_neg}: {oid_flux, temp_flux, diff_flux}\n')

(58649.35606480017, 18.88290023803711, 18.88290023803711, 18.56999969482422, -0.3129005432128906)
f: (948.0689095273461, 1264.7367029598558, 316.6677934325097)

(58654.31391200004, 17.951900482177734, 17.951900482177734, 18.56999969482422, -0.6180992126464844)
t: (2234.8058890907673, 1264.7367029598558, 970.0691861309115)

(58693.29187500011, 18.2632999420166, 18.2632999420166, 18.56999969482422, -0.3066997528076172)
t: (1677.5675748100446, 1264.7367029598558, 412.8308718501887)

(58699.20634259982, 18.40180015563965, 18.40180015563965, 18.56999969482422, -0.1681995391845703)
t: (1476.6580515285705, 1264.7367029598558, 211.92134856871462)

(58716.26269679982, 18.553600311279297, 18.553600311279297, 18.56999969482422, -0.016399383544921875)
t: (1283.9847792060416, 1264.7367029598558, 19.248076246185747)

(58723.259131900035, 18.775100708007812, 18.775100708007812, 18.56999969482422, -0.20510101318359375)
f: (1047.0314256200943, 1264.7367029598558, 217.70527733976155)

(58726.24780090013

confirmed that the mag used in antares' lightcurve is magpsf, so i will use that as the oid mag. i also see that in [this paper](https://ui.adsabs.harvard.edu/search/fq=%7B!type%3Daqp%20v%3D%24fq_database%7D&fq_database=(database%3Aastronomy%20OR%20database%3Aphysics)&q=THE%20ZWICKY%20TRANSIENT%20FACILITY%20ALERT%20DISTRIBUTION%20SYSTEM&sort=date%20desc%2C%20bibcode%20desc&p_=0), where they describe their alert system. table 1 shows some of the schema for the alerts, and it describes magnr as the "magnitude of nearest source in reference image PSF-catalog within 30 arcsec" so i think this Is whats used for the alert candidacy. 

an issue i have with this is that the isdiffpos variable doesnt always match up with that. its clear that they made that variable so the the difference mag/flux would always be a positive value for ease of calculations, but in using their convention most diff_mags are negative, but its not even consistent as some are positive. it could be that the isdiffpos value was based on the flux info, but if that were true then i think all values would be negative instead of just some. im going to calc the fluxes and see what happens. 

yea, i thought that the diff flux would be mostly positive (flipped from the diff mag) and i was right. still though, there remain diff fluxes with negative values which shouldnt be happening if this is how the isdiffpos value is determined. so, whats happening?????

idk, but im going to try rewriting my previous work to use magpsf and magnr directly as the oid and temp mags and see how that ends up.

In [268]:
def ztf_quality(alert):
    quality = True
    quality &= alert.properties['ztf_rb'] >= 0.65
    quality &= alert.properties['ztf_nbad'] == 0
    quality &= alert.properties['ztf_fwhm'] <= 5
    quality &= alert.properties['ztf_elong'] <= 1.2
    quality &= abs(alert.properties['ztf_magdiff']) <= 0.1
    return quality

def ztf_alert2(alert):
    bands = ['x', 'g', 'R', 'i']
    magzp = [0, 26.325, 26.275, 25.660]

    band_id = alert.properties['ztf_fid']
    obs_date = float(alert.properties['ant_mjd'])
    ant_mag = float(alert.properties['ant_mag'])
    oid_mag = float(alert.properties['ztf_magpsf'])
    temp_mag = float(alert.properties['ztf_magnr'])
    
    diff_mag = abs(oid_mag - pnr_mag)

    oid_snr = float(1.0857 / alert.properties['ztf_sigmapsf'])

    band = 'ztf_' + str(bands[band_id])
    return {band: [obs_date, oid_mag, pnr_mag, diff_mag, pos_or_neg, oid_snr]}

In [281]:
#this is running the filter with the ztf mag changes and a ztf quality filter, i will add a lsst quality filter later
#this is also the version that only looks at the most recent observations

oids_var2 = {}
passed = 0
for locus in search.search(q_lsst_magcut):
    oid_dv = locus.to_devkit()
    oid_locus = DevKitLocus.model_validate(oid_dv)

    high_var_oid = oid_var_check(oid_locus.alerts, 20, 0.2)

    if high_var_oid == None:
        passed += 1
        continue

    oids_var2[oid_dv['id']] = high_var_oid
    
    if len(oids_var2) >= 5:
        break
        
print(f'{passed} oids were passed\n{oids_var2}')

35 oids were passed
{'ANT2020bc5tc': {'lsst_g': 0, 'lsst_r': 0, 'lsst_i': 0, 'lsst_z': {'data':             mjd    oid_mag  template_mag  diff_mag  pos/neg        snr
0  61191.350794  17.805262     17.537825 -0.267438       -1  49.666527, 'variance': 0, 'var_diff_rat': 0}, 'lsst_u': 0, 'lsst_y': 0, 'ztf_g': {'data':               mjd    oid_mag  template_mag  diff_mag pos/neg        snr
0    58641.418715  18.992599        18.504  0.488600       f   9.908643
1    58647.375637  17.624201        18.504  0.879799       f  18.111603
2    58660.421944  17.401899        18.504  1.102100       f  21.362795
3    58667.435868  19.160900        18.504  0.656900       f   7.582181
4    58674.423877  18.273500        18.504  0.230499       f  15.952100
..            ...        ...           ...       ...     ...        ...
156  60967.211181  17.558578        18.504  0.945421       f  17.945783
157  60985.176759  18.902700        18.504  0.398701       f   4.840846
158  61199.433634  19.060989      